In [1]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [2]:
# ============================================
# PHASE 5 — COMMON LEARNING UNIT
# ============================================

In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from PIL import Image

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
PROJECT_ROOT = Path.cwd().parent

CRIC_ROOT = PROJECT_ROOT / "cric_cervix"
RIVA_ROOT = PROJECT_ROOT / "riva_1.0"

CRIC_IMAGES = CRIC_ROOT / "images"
CRIC_METADATA = CRIC_ROOT / "metadata"

RIVA_IMAGES = RIVA_ROOT / "images"
RIVA_ANNOTATIONS = RIVA_ROOT / "annotations" / "annotations.json"

print("Project root:", PROJECT_ROOT)
print("CRIC images:", CRIC_IMAGES)
print("RIVA images:", RIVA_IMAGES)
print("RIVA annotations:", RIVA_ANNOTATIONS)

Project root: c:\Users\nanda\Documents\cervical-semi-sl
CRIC images: c:\Users\nanda\Documents\cervical-semi-sl\cric_cervix\images
RIVA images: c:\Users\nanda\Documents\cervical-semi-sl\riva_1.0\images
RIVA annotations: c:\Users\nanda\Documents\cervical-semi-sl\riva_1.0\annotations\annotations.json


In [5]:
paths_to_check = {
    "CRIC images": CRIC_IMAGES,
    "CRIC metadata": CRIC_METADATA,
    "RIVA images": RIVA_IMAGES,
    "RIVA annotations": RIVA_ANNOTATIONS,
}

for name, path in paths_to_check.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

CRIC images: OK
CRIC metadata: OK
RIVA images: OK
RIVA annotations: OK


In [6]:
with open(RIVA_ANNOTATIONS, "r", encoding="utf-8") as f:
    riva_data = json.load(f)

print("Number of RIVA image records:", len(riva_data))

Number of RIVA image records: 959


In [7]:
first_key = next(iter(riva_data))

print("First image key:", first_key)
print()
print("Record structure:")
print(riva_data[first_key])

First image key: LSIL_45_3

Record structure:
{'annotator_2': [{'x': 97.70992366412213, 'y': 51.399491094147585, 'keypointlabels': 'ASCUS'}, {'x': 95.16539440203562, 'y': 52.16284987277354, 'keypointlabels': 'ASCUS'}, {'x': 82.95165394402035, 'y': 13.486005089058525, 'keypointlabels': 'ASCUS'}, {'x': 81.93384223918575, 'y': 16.28498727735369, 'keypointlabels': 'ASCUS'}, {'x': 58.524173027989825, 'y': 37.150127226463106, 'keypointlabels': 'INFL'}, {'x': 98.21882951653944, 'y': 23.918575063613233, 'keypointlabels': 'INFL'}, {'x': 59.03307888040712, 'y': 10.687022900763358, 'keypointlabels': 'INFL'}, {'x': 21.882951653944023, 'y': 3.307888040712468, 'keypointlabels': 'INFL'}, {'x': 3.5623409669211195, 'y': 24.173027989821882, 'keypointlabels': 'INFL'}, {'x': 25.190839694656486, 'y': 28.498727735368956, 'keypointlabels': 'INFL'}, {'x': 24.427480916030532, 'y': 21.374045801526716, 'keypointlabels': 'ASCUS'}, {'x': 9.669211195928753, 'y': 98.21882951653944, 'keypointlabels': 'INFL'}, {'x': 4

In [8]:
print(list(CRIC_METADATA.glob("*")))

[WindowsPath('c:/Users/nanda/Documents/cervical-semi-sl/cric_cervix/metadata/classifications.csv'), WindowsPath('c:/Users/nanda/Documents/cervical-semi-sl/cric_cervix/metadata/classifications.json'), WindowsPath('c:/Users/nanda/Documents/cervical-semi-sl/cric_cervix/metadata/README.md')]


In [9]:
cric_csv = CRIC_METADATA / "classifications.csv"

cric_df = pd.read_csv(cric_csv)

print("CRIC rows:", len(cric_df))
print()
print("CRIC columns:")
print(cric_df.columns.tolist())

CRIC rows: 11534

CRIC columns:
['image_id', 'image_filename', 'image_doi', 'cell_id', 'bethesda_system', 'nucleus_x', 'nucleus_y']


In [10]:
cric_df[["bethesda_system"]].head(10)

,bethesda_system
0,SCC
1,SCC
2,SCC
3,SCC
4,Negative for intraepithelial lesion
5,Negative for intraepithelial lesion
6,LSIL
7,LSIL
8,HSIL
9,ASC-H


In [11]:
print(cric_df.columns.tolist())

['image_id', 'image_filename', 'image_doi', 'cell_id', 'bethesda_system', 'nucleus_x', 'nucleus_y']


In [12]:
record = riva_data[first_key]

print(type(record))

if isinstance(record, dict):
    print("Top-level keys:")
    for key in record.keys():
        print(" -", key)

<class 'dict'>
Top-level keys:
 - annotator_2


In [13]:
for key, value in record.items():
    print("\nKEY:", key)
    print("TYPE:", type(value))
    
    if isinstance(value, list):
        print("LENGTH:", len(value))
        if len(value) > 0:
            print("FIRST ITEM:", value[0])
    elif isinstance(value, dict):
        print("SUBKEYS:", list(value.keys())[:20])


KEY: annotator_2
TYPE: <class 'list'>
LENGTH: 17
FIRST ITEM: {'x': 97.70992366412213, 'y': 51.399491094147585, 'keypointlabels': 'ASCUS'}


# Phase 5 — Common Learning Unit

## CRIC sample-unit inspection

CRIC provides cell-level annotations through:
- image_id
- image_filename
- cell_id
- nucleus_x
- nucleus_y
- bethesda_system

The candidate learning unit is one annotated cell represented by a cell-centered crop around its nucleus coordinate.

In [14]:
print("CRIC sample columns:")
print(cric_df[
    [
        "image_id",
        "image_filename",
        "cell_id",
        "bethesda_system",
        "nucleus_x",
        "nucleus_y"
    ]
].head(10).to_string(index=False))

CRIC sample columns:
 image_id                       image_filename  cell_id                     bethesda_system  nucleus_x  nucleus_y
      400 9ae8a4edde40219bad6303cebc672ee4.png        1                                 SCC        792        462
      400 9ae8a4edde40219bad6303cebc672ee4.png        2                                 SCC        601        678
      400 9ae8a4edde40219bad6303cebc672ee4.png        3                                 SCC        363        467
      400 9ae8a4edde40219bad6303cebc672ee4.png        4                                 SCC        599        437
      400 9ae8a4edde40219bad6303cebc672ee4.png        5 Negative for intraepithelial lesion       1186        450
      400 9ae8a4edde40219bad6303cebc672ee4.png        6 Negative for intraepithelial lesion        408        599
      400 9ae8a4edde40219bad6303cebc672ee4.png        7                                LSIL        483        293
      400 9ae8a4edde40219bad6303cebc672ee4.png        8            

In [15]:
print("CRIC X coordinate range:")
print(cric_df["nucleus_x"].min(), "to", cric_df["nucleus_x"].max())

print("\nCRIC Y coordinate range:")
print(cric_df["nucleus_y"].min(), "to", cric_df["nucleus_y"].max())

print("\nMissing coordinates:")
print(cric_df[["nucleus_x", "nucleus_y"]].isna().sum())

CRIC X coordinate range:
3 to 1278

CRIC Y coordinate range:
1 to 1019

Missing coordinates:
nucleus_x    0
nucleus_y    0
dtype: int64


In [16]:
cric_cells_per_image = (
    cric_df.groupby("image_id")
    .size()
    .sort_values(ascending=False)
)

print("Number of CRIC images with annotations:", len(cric_cells_per_image))
print("\nCells per image:")
print(cric_cells_per_image.describe())

Number of CRIC images with annotations: 400

Cells per image:
count    400.000000
mean      28.835000
std       21.916858
min        2.000000
25%       13.750000
50%       22.000000
75%       39.250000
max      136.000000
dtype: float64


In [17]:
print("\nImages with the fewest cells:")
print(cric_cells_per_image.sort_values().head(10))

print("\nImages with the most cells:")
print(cric_cells_per_image.head(10))


Images with the fewest cells:
image_id
219    2
353    3
317    3
369    4
34     4
175    4
105    4
207    4
380    4
241    5
dtype: int64

Images with the most cells:
image_id
339    136
73     123
395    113
396    106
71     104
384    102
68      97
371     94
70      93
397     91
dtype: int64


In [18]:
row = cric_df.iloc[0]

image_path = CRIC_IMAGES / row["image_filename"]

print("Image:", image_path)
print("Exists:", image_path.exists())

if image_path.exists():
    with Image.open(image_path) as img:
        print("Image size:", img.size)
        print("Image mode:", img.mode)

print("\nCell coordinate:")
print("x =", row["nucleus_x"])
print("y =", row["nucleus_y"])

Image: c:\Users\nanda\Documents\cervical-semi-sl\cric_cervix\images\9ae8a4edde40219bad6303cebc672ee4.png
Exists: True


Image size: (1376, 1020)
Image mode: RGB

Cell coordinate:
x = 792
y = 462


In [19]:
riva_image_path = RIVA_IMAGES / f"{first_key}.png"

print("RIVA image:", riva_image_path)
print("Exists:", riva_image_path.exists())

if riva_image_path.exists():
    with Image.open(riva_image_path) as img:
        print("Image size:", img.size)
        print("Image mode:", img.mode)

RIVA image: c:\Users\nanda\Documents\cervical-semi-sl\riva_1.0\images\LSIL_45_3.png
Exists: True
Image size: (1024, 1024)
Image mode: RGB


In [20]:
riva_points = []

for annotator, annotations in riva_data[first_key].items():
    for ann in annotations:
        riva_points.append({
            "image_id": first_key,
            "annotator": annotator,
            "x": ann.get("x"),
            "y": ann.get("y"),
            "original_label": ann.get("keypointlabels")
        })

riva_points_df = pd.DataFrame(riva_points)

print(riva_points_df.to_string(index=False))

 image_id   annotator         x         y original_label
LSIL_45_3 annotator_2 97.709924 51.399491          ASCUS
LSIL_45_3 annotator_2 95.165394 52.162850          ASCUS
LSIL_45_3 annotator_2 82.951654 13.486005          ASCUS
LSIL_45_3 annotator_2 81.933842 16.284987          ASCUS
LSIL_45_3 annotator_2 58.524173 37.150127           INFL
LSIL_45_3 annotator_2 98.218830 23.918575           INFL
LSIL_45_3 annotator_2 59.033079 10.687023           INFL
LSIL_45_3 annotator_2 21.882952  3.307888           INFL
LSIL_45_3 annotator_2  3.562341 24.173028           INFL
LSIL_45_3 annotator_2 25.190840 28.498728           INFL
LSIL_45_3 annotator_2 24.427481 21.374046          ASCUS
LSIL_45_3 annotator_2  9.669211 98.218830           INFL
LSIL_45_3 annotator_2 47.837150 66.666667          ASCUS
LSIL_45_3 annotator_2 91.348601 45.801527          ASCUS
LSIL_45_3 annotator_2 94.656489 47.073791          ASCUS
LSIL_45_3 annotator_2 87.022901 58.015267          ASCUS
LSIL_45_3 annotator_2 91.348601

In [21]:
print("RIVA X range:")
print(riva_points_df["x"].min(), "to", riva_points_df["x"].max())

print("\nRIVA Y range:")
print(riva_points_df["y"].min(), "to", riva_points_df["y"].max())

print("\nMissing values:")
print(
    riva_points_df[
        ["x", "y", "original_label"]
    ].isna().sum()
)

RIVA X range:
3.5623409669211195 to 98.21882951653944

RIVA Y range:
3.307888040712468 to 98.21882951653944

Missing values:
x                 0
y                 0
original_label    0
dtype: int64


In [22]:
annotator_summary = []

for image_id, record in riva_data.items():
    for annotator, annotations in record.items():
        annotator_summary.append({
            "image_id": image_id,
            "annotator": annotator,
            "num_annotations": len(annotations)
        })

annotator_df = pd.DataFrame(annotator_summary)

print(annotator_df.head())
print("\nAnnotator counts:")
print(annotator_df["annotator"].value_counts())

    image_id    annotator  num_annotations
0  LSIL_45_3  annotator_2               17
1   LSIL_7_7  annotator_1               30
2   LSIL_7_7  annotator_2               25
3   LSIL_7_7  annotator_3               24
4   LSIL_7_7  annotator_4               22

Annotator counts:
annotator
annotator_4    535
annotator_3    530
annotator_1    529
annotator_2    523
Name: count, dtype: int64


In [23]:
patch_annotator_counts = (
    annotator_df
    .groupby("image_id")["annotator"]
    .nunique()
)

print("\nNumber of annotators per RIVA image:")
print(patch_annotator_counts.value_counts().sort_index())


Number of annotators per RIVA image:
annotator
1    573
4    386
Name: count, dtype: int64


In [24]:
# Search CRIC metadata columns for patient/subject/study/slide identifiers

keywords = [
    "patient",
    "subject",
    "person",
    "case",
    "study",
    "slide",
    "specimen"
]

matching_columns = [
    col for col in cric_df.columns
    if any(keyword in col.lower() for keyword in keywords)
]

print("Potential patient/subject/slide columns:")
print(matching_columns)

Potential patient/subject/slide columns:
[]


In [25]:
print("Unique image IDs:", cric_df["image_id"].nunique())
print("Unique image filenames:", cric_df["image_filename"].nunique())
print("Unique image DOIs:", cric_df["image_doi"].nunique())

print("\nSample image metadata:")
print(
    cric_df[
        ["image_id", "image_filename", "image_doi"]
    ].drop_duplicates().head(20).to_string(index=False)
)

Unique image IDs: 400
Unique image filenames: 400
Unique image DOIs: 400

Sample image metadata:
 image_id                       image_filename                    image_doi
      400 9ae8a4edde40219bad6303cebc672ee4.png 10.6084/m9.figshare.12230906
      399 dc2df7c3f88649ded343b13b9486cddf.png 10.6084/m9.figshare.12230903
      398 0cc6e576dac3046d279bba515e10a7a9.png 10.6084/m9.figshare.12230900
      397 dbf4ff8985d6e352b46cd8f42add72e8.png 10.6084/m9.figshare.12230897
      396 ba9d94a8fdbaba01220db89cde170b24.png 10.6084/m9.figshare.12230894
      395 1a1319ef78624dc20d4a43cadde1da01.png 10.6084/m9.figshare.12230891
      394 f1662645f23a20ea9806ca8c6d655fba.png 10.6084/m9.figshare.12230888
      393 363b6b00d925e5c52694b8f7b678c53b.png 10.6084/m9.figshare.12230885
      392 ac20a24b37b7e87dcc375e6b7420d40e.png 10.6084/m9.figshare.12230879
      391 0d07c6d5b42ace6162cff25fc06b21b9.png 10.6084/m9.figshare.12230873
      390 1208a64963167c6b7d56a072f8b8ee1a.png 10.6084/m9.figshare.

In [26]:
readme_path = CRIC_METADATA / "README.md"

print("README exists:", readme_path.exists())

if readme_path.exists():
    readme_text = readme_path.read_text(encoding="utf-8", errors="replace")
    
    for line in readme_text.splitlines():
        lower = line.lower()
        if any(
            keyword in lower
            for keyword in [
                "patient",
                "subject",
                "case",
                "slide",
                "specimen",
                "image"
            ]
        ):
            print(line)

README exists: True
400 images from microscope slides of the uterine cervix using the conventional smear (Pap smear) and the epithelial cell abnormalities classified according to Bethesda system.
- `image_id`
  This is the integer that identifies the image at http://database.cric.com.br/.
- `image_filename`
  This is the name that identifies the image in the ZIP file that you have.
- `image_doi`
  This is the DOI that identifies the image.


In [27]:
riva_ids = list(riva_data.keys())

print("First 30 RIVA IDs:")
for image_id in riva_ids[:30]:
    print(image_id)

First 30 RIVA IDs:
LSIL_45_3
LSIL_7_7
HSIL_11_5
HSIL_22_6
LSIL_35_1
LSIL_41_5
HSIL_22_10
LSIL_29_1
HSIL_1_18
LSIL_28_7
HSIL_LSIL_2_6
ASCUS_1_17
LSIL_52_5
HSIL_18_13
HSIL_LSIL_1_11
HSIL_28_1
HSIL_24_3
HSIL_12_3
LSIL_24_1
LSIL_7_10
LSIL_9_1
SCC_3_1
ASCUS_6_1
LSIL_32_6
LSIL_28_6
ASCH_2_3
LSIL_38_9
SCC_1_2
HSIL_14_14
LSIL_28_15


In [28]:
documentation_root = PROJECT_ROOT / "documentation"

for path in documentation_root.rglob("*riva*.md"):
    print(path)

c:\Users\nanda\Documents\cervical-semi-sl\documentation\stage_01_dataset_archaeology\riva_initial_findings.md
c:\Users\nanda\Documents\cervical-semi-sl\documentation\stage_03_dataset_inspection\riva_annotation_structure.md


In [29]:
# ============================================================
# PHASE 5 — RIVA IDENTIFIER / PATIENT STRUCTURE CHECK
# ============================================================

print("=" * 70)
print("1. RIVA IMAGE IDENTIFIERS")
print("=" * 70)

riva_ids = list(riva_data.keys())

print("Total RIVA image/annotation records:", len(riva_ids))
print("\nFirst 30 RIVA IDs:")
for image_id in riva_ids[:30]:
    print(" ", image_id)


print("\n" + "=" * 70)
print("2. RIVA ID STRUCTURE")
print("=" * 70)

# Split IDs into underscore-separated components
id_parts = [image_id.split("_") for image_id in riva_ids]

print("Examples of ID components:")
for image_id, parts in list(zip(riva_ids, id_parts))[:20]:
    print(f"{image_id:25s} -> {parts}")

print("\nNumber of components per ID:")
component_counts = pd.Series([len(parts) for parts in id_parts])
print(component_counts.value_counts().sort_index())


print("\n" + "=" * 70)
print("3. POSSIBLE CASE / PATIENT GROUPING")
print("=" * 70)

# Show how many images share the first component
first_components = [parts[0] for parts in id_parts]

first_component_counts = (
    pd.Series(first_components)
    .value_counts()
    .sort_index()
)

print("Unique first components:", len(first_component_counts))
print("\nFirst-component counts:")
print(first_component_counts.to_string())


print("\n" + "=" * 70)
print("4. LOCAL RIVA DOCUMENTATION")
print("=" * 70)

documentation_root = PROJECT_ROOT / "documentation"

riva_docs = list(documentation_root.rglob("*riva*.md"))

print("RIVA-related documentation files found:")
for path in riva_docs:
    print(" ", path.relative_to(PROJECT_ROOT))

print("\nSearching documentation for patient/case/subject/slide terminology...")

keywords = [
    "patient",
    "subject",
    "case",
    "slide",
    "specimen",
    "sample",
    "image",
    "patch"
]

for path in riva_docs:
    print("\n---", path.relative_to(PROJECT_ROOT), "---")

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    matched_lines = []

    for line in text.splitlines():
        lower = line.lower()

        if any(keyword in lower for keyword in keywords):
            matched_lines.append(line.strip())

    if matched_lines:
        for line in matched_lines[:80]:
            print(line)
    else:
        print("No matching terminology found.")


print("\n" + "=" * 70)
print("5. SUMMARY")
print("=" * 70)

print("CRIC patient/subject metadata: NOT FOUND")
print("CRIC safest grouping unit currently available: IMAGE")
print("RIVA ID structure: printed above")
print("RIVA documentation evidence: printed above")

1. RIVA IMAGE IDENTIFIERS
Total RIVA image/annotation records: 959

First 30 RIVA IDs:
  LSIL_45_3
  LSIL_7_7
  HSIL_11_5
  HSIL_22_6
  LSIL_35_1
  LSIL_41_5
  HSIL_22_10
  LSIL_29_1
  HSIL_1_18
  LSIL_28_7
  HSIL_LSIL_2_6
  ASCUS_1_17
  LSIL_52_5
  HSIL_18_13
  HSIL_LSIL_1_11
  HSIL_28_1
  HSIL_24_3
  HSIL_12_3
  LSIL_24_1
  LSIL_7_10
  LSIL_9_1
  SCC_3_1
  ASCUS_6_1
  LSIL_32_6
  LSIL_28_6
  ASCH_2_3
  LSIL_38_9
  SCC_1_2
  HSIL_14_14
  LSIL_28_15

2. RIVA ID STRUCTURE
Examples of ID components:
LSIL_45_3                 -> ['LSIL', '45', '3']
LSIL_7_7                  -> ['LSIL', '7', '7']
HSIL_11_5                 -> ['HSIL', '11', '5']
HSIL_22_6                 -> ['HSIL', '22', '6']
LSIL_35_1                 -> ['LSIL', '35', '1']
LSIL_41_5                 -> ['LSIL', '41', '5']
HSIL_22_10                -> ['HSIL', '22', '10']
LSIL_29_1                 -> ['LSIL', '29', '1']
HSIL_1_18                 -> ['HSIL', '1', '18']
LSIL_28_7                 -> ['LSIL', '28', '7']
HSIL_LS

In [30]:
# ============================================================
# PHASE 5 — RIVA STRUCTURE + COORDINATE + ANNOTATOR CHECK
# ============================================================

print("=" * 75)
print("RIVA DOCUMENTATION DEEP CHECK")
print("=" * 75)

search_terms = [
    "patient",
    "subject",
    "case",
    "slide",
    "specimen",
    "sample",
    "identifier",
    "coordinate",
    "pixel",
    "percentage",
    "normalized",
    "annotator",
    "cluster",
    "meanshift",
    "MeanShift",
    "consensus"
]

for path in riva_docs:

    print("\n" + "=" * 75)
    print("FILE:", path.relative_to(PROJECT_ROOT))
    print("=" * 75)

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i, line in enumerate(lines):

        lower = line.lower()

        if any(term.lower() in lower for term in search_terms):

            start = max(0, i - 2)
            end = min(len(lines), i + 3)

            print(f"\n--- lines {start + 1}-{end} ---")

            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")


print("\n" + "=" * 75)
print("RIVA ANNOTATOR STRUCTURE FROM RAW JSON")
print("=" * 75)

annotator_counts = {}

for image_id, record in riva_data.items():

    for annotator, annotations in record.items():

        annotator_counts.setdefault(annotator, {
            "images": 0,
            "annotations": 0
        })

        annotator_counts[annotator]["images"] += 1
        annotator_counts[annotator]["annotations"] += len(annotations)

print("\nAnnotator summary:")

for annotator, values in sorted(annotator_counts.items()):
    print(
        f"{annotator}: "
        f"{values['images']} images, "
        f"{values['annotations']} annotations"
    )


print("\n" + "=" * 75)
print("SAMPLE RIVA COORDINATES")
print("=" * 75)

for image_id in riva_ids[:5]:

    record = riva_data[image_id]

    print(f"\n{image_id}")

    for annotator, annotations in record.items():

        print(f"  {annotator}: {len(annotations)} annotations")

        for ann in annotations[:3]:

            print(
                "   ",
                {
                    "x": ann.get("x"),
                    "y": ann.get("y"),
                    "label": ann.get("keypointlabels")
                }
            )


print("\n" + "=" * 75)
print("END OF CHECK")
print("=" * 75)

RIVA DOCUMENTATION DEEP CHECK

FILE: documentation\stage_01_dataset_archaeology\riva_initial_findings.md

--- lines 21-25 ---
21: The `annotations.json` file contains a JSON dictionary.
22: 
23: Each top-level key appears to identify an individual sample/image. Examples observed include:
24: 
25: - `LSIL_45_3`

--- lines 30-34 ---
30: - `NILM_1_1`
31: 
32: Each sample can contain annotations from one or more annotators.
33: 
34: Examples observed include:

--- lines 34-38 ---
34: Examples observed include:
35: 
36: - `annotator_1`
37: - `annotator_2`
38: 

--- lines 35-39 ---
35: 
36: - `annotator_1`
37: - `annotator_2`
38: 
39: ## Initial observations

--- lines 47-51 ---
47: - `keypointlabels`
48: 
49: The `x` and `y` values appear to be normalized coordinates rather than raw pixel coordinates, since observed values range approximately from 0 to 100.
50: 
51: Observed annotation labels include:

--- lines 55-59 ---
55: - `INFL`
56: 
57: Additional diagnostic categories are also repre

In [31]:
# ============================================================
# PHASE 5 — RIVA COORDINATE VERIFICATION
# ============================================================

all_x = []
all_y = []
missing_x = 0
missing_y = 0
missing_labels = 0

label_counts = {}

for image_id, record in riva_data.items():

    for annotator, annotations in record.items():

        for ann in annotations:

            x = ann.get("x")
            y = ann.get("y")
            label = ann.get("keypointlabels")

            if x is None:
                missing_x += 1
            else:
                all_x.append(float(x))

            if y is None:
                missing_y += 1
            else:
                all_y.append(float(y))

            if label is None:
                missing_labels += 1
            else:
                label_counts[label] = label_counts.get(label, 0) + 1


print("=" * 70)
print("RIVA GLOBAL COORDINATE CHECK")
print("=" * 70)

print("Total x coordinates:", len(all_x))
print("Total y coordinates:", len(all_y))

print("\nX range:")
print("  min:", min(all_x))
print("  max:", max(all_x))

print("\nY range:")
print("  min:", min(all_y))
print("  max:", max(all_y))

print("\nCoordinates outside [0, 100]:")
print("  X:", sum(x < 0 or x > 100 for x in all_x))
print("  Y:", sum(y < 0 or y > 100 for y in all_y))

print("\nMissing values:")
print("  X:", missing_x)
print("  Y:", missing_y)
print("  Labels:", missing_labels)


print("\n" + "=" * 70)
print("RIVA LABEL COUNTS")
print("=" * 70)

for label, count in sorted(label_counts.items()):
    print(f"{label:8s}: {count}")


print("\n" + "=" * 70)
print("EXAMPLE PERCENTAGE → PIXEL CONVERSION")
print("=" * 70)

IMAGE_SIZE = 1024

example_coordinates = [
    (0, 0),
    (25, 25),
    (50, 50),
    (75, 75),
    (100, 100)
]

for x, y in example_coordinates:

    pixel_x = x / 100 * IMAGE_SIZE
    pixel_y = y / 100 * IMAGE_SIZE

    print(
        f"({x:>3}, {y:>3}) "
        f"→ "
        f"({pixel_x:>7.2f}, {pixel_y:>7.2f}) pixels"
    )


print("\n" + "=" * 70)
print("END")
print("=" * 70)

RIVA GLOBAL COORDINATE CHECK
Total x coordinates: 26158
Total y coordinates: 26158

X range:
  min: 0.1295336787564767
  max: 99.74554707379136

Y range:
  min: 0.0
  max: 99.23664122137404

Coordinates outside [0, 100]:
  X: 0
  Y: 0

Missing values:
  X: 0
  Y: 0
  Labels: 0

RIVA LABEL COUNTS
ASCH    : 416
ASCUS   : 356
ENDO    : 1270
HSIL    : 1835
INFL    : 8190
LSIL    : 3048
NILM    : 9457
SCC     : 1586

EXAMPLE PERCENTAGE → PIXEL CONVERSION
(  0,   0) → (   0.00,    0.00) pixels
( 25,  25) → ( 256.00,  256.00) pixels
( 50,  50) → ( 512.00,  512.00) pixels
( 75,  75) → ( 768.00,  768.00) pixels
(100, 100) → (1024.00, 1024.00) pixels

END


In [32]:
# ============================================================
# PHASE 5 — RIVA ANNOTATION OVERLAP / CLUSTERING CHECK
# ============================================================

print("=" * 75)
print("SEARCHING LOCAL DOCUMENTATION FOR RIVA CLUSTERING / CONSENSUS")
print("=" * 75)

clustering_terms = [
    "cluster",
    "clustering",
    "MeanShift",
    "Mean Shift",
    "meanshift",
    "consensus",
    "annotator",
    "ground truth",
    "ground-truth",
    "agreement",
    "overlap",
    "same cell",
    "cell"
]

for path in riva_docs:

    print("\n" + "=" * 75)
    print("FILE:", path.relative_to(PROJECT_ROOT))
    print("=" * 75)

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    printed = set()

    for i, line in enumerate(lines):

        lower = line.lower()

        if any(term.lower() in lower for term in clustering_terms):

            # Show surrounding context
            start = max(0, i - 3)
            end = min(len(lines), i + 4)

            # Avoid printing the exact same context repeatedly
            context_key = (start, end)

            if context_key in printed:
                continue

            printed.add(context_key)

            print(f"\n--- lines {start + 1}-{end} ---")

            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")


print("\n" + "=" * 75)
print("RAW RIVA OVERLAP EXAMPLE")
print("=" * 75)

# Compare annotations from different annotators for one
# multi-annotator sample.
example_id = "LSIL_7_7"

if example_id in riva_data:

    record = riva_data[example_id]

    print(f"Sample: {example_id}")
    print("Image size: 1024 x 1024")
    print()

    for annotator, annotations in record.items():

        print(f"{annotator}: {len(annotations)} annotations")

        for ann in annotations[:10]:

            x = ann["x"]
            y = ann["y"]
            label = ann["keypointlabels"]

            px = x / 100 * 1024
            py = y / 100 * 1024

            print(
                f"  normalized=({x:.2f}, {y:.2f}) "
                f"pixel=({px:.1f}, {py:.1f}) "
                f"label={label}"
            )

else:
    print(f"{example_id} not found.")


print("\n" + "=" * 75)
print("END")
print("=" * 75)

SEARCHING LOCAL DOCUMENTATION FOR RIVA CLUSTERING / CONSENSUS

FILE: documentation\stage_01_dataset_archaeology\riva_initial_findings.md

--- lines 29-35 ---
29: - `ASCUS_1_17`
30: - `NILM_1_1`
31: 
32: Each sample can contain annotations from one or more annotators.
33: 
34: Examples observed include:
35: 

--- lines 33-39 ---
33: 
34: Examples observed include:
35: 
36: - `annotator_1`
37: - `annotator_2`
38: 
39: ## Initial observations

--- lines 34-40 ---
34: Examples observed include:
35: 
36: - `annotator_1`
37: - `annotator_2`
38: 
39: ## Initial observations
40: 

--- lines 63-69 ---
63: 
64: ## Important caution
65: 
66: The diagnostic prefix in a sample identifier has NOT yet been formally established as the ground-truth classification field for our experiments.
67: 
68: The relationship between:
69: 

--- lines 69-75 ---
69: 
70: 1. the sample/image identifier,
71: 2. the annotation labels,
72: 3. the annotators, and
73: 4. the final dataset-level ground truth
74: 
75: must

In [33]:
# Phase 5 — RIVA annotation structure inspection

from collections import Counter, defaultdict

# Load RIVA annotations
with open(RIVA_ANNOTATIONS, "r", encoding="utf-8") as f:
    riva_data = json.load(f)

print("Number of RIVA image records:", len(riva_data))

# Count annotations by annotator and label
annotator_counts = Counter()
label_counts = Counter()

# Store a flat representation for later clustering
riva_annotations = []

for image_id, image_data in riva_data.items():

    for annotator, annotations in image_data.items():

        # Ignore unexpected non-annotator fields if present
        if not annotator.startswith("annotator_"):
            continue

        for idx, ann in enumerate(annotations):

            x = ann.get("x")
            y = ann.get("y")
            label = ann.get("label")

            riva_annotations.append({
                "image_id": image_id,
                "annotator": annotator,
                "annotation_id": idx,
                "x_norm": x,
                "y_norm": y,
                "label": label
            })

            annotator_counts[annotator] += 1
            label_counts[label] += 1

riva_ann_df = pd.DataFrame(riva_annotations)

print("\nFlat RIVA annotation table:")
print(riva_ann_df.shape)

print("\nColumns:")
print(riva_ann_df.columns.tolist())

print("\nAnnotations by annotator:")
print(annotator_counts)

print("\nAnnotations by label:")
print(label_counts)

print("\nMissing values:")
print(riva_ann_df[["x_norm", "y_norm", "label"]].isna().sum())

print("\nUnique images represented:")
print(riva_ann_df["image_id"].nunique())

print("\nTotal annotations:")
print(len(riva_ann_df))

Number of RIVA image records: 959

Flat RIVA annotation table:
(26158, 6)

Columns:
['image_id', 'annotator', 'annotation_id', 'x_norm', 'y_norm', 'label']

Annotations by annotator:
Counter({'annotator_1': 8171, 'annotator_2': 7440, 'annotator_3': 5766, 'annotator_4': 4781})

Annotations by label:
Counter({None: 26158})

Missing values:
x_norm        0
y_norm        0
label     26158
dtype: int64

Unique images represented:
959

Total annotations:
26158


In [34]:
# Inspect the exact structure of one RIVA annotation

for image_id, image_data in riva_data.items():
    print("Image ID:", image_id)

    for annotator, annotations in image_data.items():
        if annotator.startswith("annotator_") and annotations:
            print("Annotator:", annotator)
            print("Number of annotations:", len(annotations))

            print("\nFirst annotation:")
            print(annotations[0])

            print("\nKeys in annotation:")
            print(annotations[0].keys())

            break

    break

Image ID: LSIL_45_3
Annotator: annotator_2
Number of annotations: 17

First annotation:
{'x': 97.70992366412213, 'y': 51.399491094147585, 'keypointlabels': 'ASCUS'}

Keys in annotation:
dict_keys(['x', 'y', 'keypointlabels'])


In [35]:
# Phase 5 — Build corrected RIVA annotation table

riva_annotations = []

for image_id, image_data in riva_data.items():

    for annotator, annotations in image_data.items():

        if not annotator.startswith("annotator_"):
            continue

        for idx, ann in enumerate(annotations):

            riva_annotations.append({
                "image_id": image_id,
                "annotator": annotator,
                "annotation_id": idx,
                "x_norm": ann["x"],
                "y_norm": ann["y"],
                "label": ann["keypointlabels"]
            })

riva_ann_df = pd.DataFrame(riva_annotations)

print("Shape:", riva_ann_df.shape)

print("\nAnnotations by annotator:")
print(riva_ann_df["annotator"].value_counts())

print("\nAnnotations by label:")
print(riva_ann_df["label"].value_counts())

print("\nMissing values:")
print(riva_ann_df[["x_norm", "y_norm", "label"]].isna().sum())

print("\nUnique images:")
print(riva_ann_df["image_id"].nunique())

print("\nTotal annotations:")
print(len(riva_ann_df))

Shape: (26158, 6)

Annotations by annotator:
annotator
annotator_1    8171
annotator_2    7440
annotator_3    5766
annotator_4    4781
Name: count, dtype: int64

Annotations by label:
label
NILM     9457
INFL     8190
LSIL     3048
HSIL     1835
SCC      1586
ENDO     1270
ASCH      416
ASCUS     356
Name: count, dtype: int64

Missing values:
x_norm    0
y_norm    0
label     0
dtype: int64

Unique images:
959

Total annotations:
26158


In [36]:
# Phase 5 — Search local RIVA dataset for final/processed clustering files

from pathlib import Path

print("RIVA directory:", RIVA_ROOT)

for path in sorted(RIVA_ROOT.rglob("*")):
    if path.is_file():
        name = path.name.lower()

        if any(term in name for term in [
            "cluster",
            "consensus",
            "processed",
            "annotation",
            "label"
        ]):
            print(path.relative_to(RIVA_ROOT))

RIVA directory: c:\Users\nanda\Documents\cervical-semi-sl\riva_1.0
annotations\annotations.json


In [37]:
from pathlib import Path

OFFICIAL_RIVA = PROJECT_ROOT / "riva_official"

print("Official RIVA repository:", OFFICIAL_RIVA.exists())

processed_files = list(OFFICIAL_RIVA.rglob("processed.csv"))

print("\nprocessed.csv files found:")
for f in processed_files:
    print(f)

print("\nNumber found:", len(processed_files))

Official RIVA repository: True

processed.csv files found:
c:\Users\nanda\Documents\cervical-semi-sl\riva_official\Raw annotations and clustering\processed_annotations\processed.csv
c:\Users\nanda\Documents\cervical-semi-sl\riva_official\Generate Annotation FIles\save_annotations\processed.csv

Number found: 2


In [38]:
# Load the official RIVA processed cell-level dataset

OFFICIAL_PROCESSED = (
    PROJECT_ROOT
    / "riva_official"
    / "Raw annotations and clustering"
    / "processed_annotations"
    / "processed.csv"
)

riva_processed_df = pd.read_csv(OFFICIAL_PROCESSED)

print("Shape:", riva_processed_df.shape)
print("\nColumns:")
print(riva_processed_df.columns.tolist())

print("\nFirst 5 rows:")
display(riva_processed_df.head())

print("\nMissing values:")
display(riva_processed_df.isna().sum())

print("\nClass counts:")
for col in ["class_annotated", "class_bethesda"]:
    if col in riva_processed_df.columns:
        print(f"\n{col}:")
        print(riva_processed_df[col].value_counts(dropna=False))

Shape: (15949, 8)

Columns:
['image_filename', 'annotator', 'annotator_id', 'nucleus_x', 'nucleus_y', 'class_annotated', 'class_bethesda', 'cluster_idx']

First 5 rows:


,image_filename,annotator,annotator_id,nucleus_x,nucleus_y,class_annotated,class_bethesda,cluster_idx
0,LSIL_45_3.png,Annotator 2,Annotator 2,1000.549618,526.330789,ASCUS,ASCUS,40000.0
1,LSIL_45_3.png,Annotator 2,Annotator 2,974.493639,534.147583,ASCUS,ASCUS,40001.0
2,LSIL_45_3.png,Annotator 2,Annotator 2,849.424936,138.096692,ASCUS,ASCUS,40002.0
3,LSIL_45_3.png,Annotator 2,Annotator 2,839.002545,166.758270,ASCUS,ASCUS,40003.0
4,LSIL_45_3.png,Annotator 2,Annotator 2,599.287532,380.417303,INFL,INFL,40004.0



Missing values:


image_filename     0
annotator          0
annotator_id       0
nucleus_x          0
nucleus_y          0
class_annotated    0
class_bethesda     0
cluster_idx        0
dtype: int64


Class counts:

class_annotated:
class_annotated
Sin lesion    5028
INFL          4715
LSIL          2002
CA            1454
HSIL          1394
ENDO           774
ASCH           361
ASCUS          221
Name: count, dtype: int64

class_bethesda:
class_bethesda
NILM     7400
INFL     2763
LSIL     2002
CA       1454
HSIL     1394
ASCH      361
ENDO      354
ASCUS     221
Name: count, dtype: int64


In [39]:
# Inspect the actual RIVA processed labels and cluster structure

print("Unique class_annotated:")
print(sorted(riva_processed_df["class_annotated"].dropna().unique()))

print("\nUnique class_bethesda:")
print(sorted(riva_processed_df["class_bethesda"].dropna().unique()))

print("\nNumber of unique images:")
print(riva_processed_df["image_filename"].nunique())

print("\nNumber of unique clusters:")
print(riva_processed_df["cluster_idx"].nunique())

print("\nAnnotator distribution:")
print(riva_processed_df["annotator"].value_counts(dropna=False))

print("\nRows per image:")
print(riva_processed_df.groupby("image_filename").size().describe())

Unique class_annotated:
['ASCH', 'ASCUS', 'CA', 'ENDO', 'HSIL', 'INFL', 'LSIL', 'Sin lesion']

Unique class_bethesda:
['ASCH', 'ASCUS', 'CA', 'ENDO', 'HSIL', 'INFL', 'LSIL', 'NILM']

Number of unique images:
959

Number of unique clusters:
15949

Annotator distribution:
annotator
Annotator 1    2565
15             2556
Annotator 2    2537
14             2463
Annotator 3    1736
Annotator 4    1604
8              1258
5              1230
Name: count, dtype: int64

Rows per image:


count    959.000000
mean      16.630865
std       12.529020
min        1.000000
25%        8.000000
50%       14.000000
75%       22.000000
max      100.000000
dtype: float64


In [40]:
# Inspect the official RIVA processed labels
# before applying the already-defined Phase 4 binary mapping

print("class_bethesda counts:")
print(riva_processed_df["class_bethesda"].value_counts())

print("\nclass_annotated counts:")
print(riva_processed_df["class_annotated"].value_counts())

print("\nBethesda × Annotated cross-tabulation:")
display(
    pd.crosstab(
        riva_processed_df["class_bethesda"],
        riva_processed_df["class_annotated"]
    )
)

class_bethesda counts:
class_bethesda
NILM     7400
INFL     2763
LSIL     2002
CA       1454
HSIL     1394
ASCH      361
ENDO      354
ASCUS     221
Name: count, dtype: int64

class_annotated counts:
class_annotated
Sin lesion    5028
INFL          4715
LSIL          2002
CA            1454
HSIL          1394
ENDO           774
ASCH           361
ASCUS          221
Name: count, dtype: int64

Bethesda × Annotated cross-tabulation:


class_annotated,ASCH,ASCUS,CA,ENDO,HSIL,INFL,LSIL,Sin lesion
class_bethesda,,,,,,,,
ASCH,361,0,0,0,0,0,0,0
ASCUS,0,221,0,0,0,0,0,0
CA,0,0,1454,0,0,0,0,0
ENDO,0,0,0,354,0,0,0,0
HSIL,0,0,0,0,1394,0,0,0
INFL,0,0,0,0,0,2763,0,0
LSIL,0,0,0,0,0,0,2002,0
NILM,0,0,0,420,0,1952,0,5028


In [41]:
pd.crosstab(
    riva_processed_df["class_bethesda"],
    riva_processed_df["class_annotated"],
    margins=True
)

class_annotated,ASCH,ASCUS,CA,ENDO,HSIL,INFL,LSIL,Sin lesion,All
class_bethesda,,,,,,,,,
ASCH,361,0,0,0,0,0,0,0,361
ASCUS,0,221,0,0,0,0,0,0,221
CA,0,0,1454,0,0,0,0,0,1454
ENDO,0,0,0,354,0,0,0,0,354
HSIL,0,0,0,0,1394,0,0,0,1394
INFL,0,0,0,0,0,2763,0,0,2763
LSIL,0,0,0,0,0,0,2002,0,2002
NILM,0,0,0,420,0,1952,0,5028,7400
All,361,221,1454,774,1394,4715,2002,5028,15949


In [42]:
# Stage 05 — Create standardized RIVA learning-unit table

riva_learning_df = riva_processed_df[
    [
        "image_filename",
        "nucleus_x",
        "nucleus_y",
        "class_bethesda",
        "class_annotated",
        "cluster_idx"
    ]
].copy()

# Keep the official Bethesda label unchanged.
# Apply the Phase 4 binary mapping to the official processed representation.
#
# RIVA's official processed file uses "CA" as its carcinoma category.
# We treat it as abnormal for our binary task.

RIVA_BINARY_MAP_PHASE4 = {
    "NILM": 0,
    "ENDO": 0,
    "INFL": 0,
    "ASCUS": 1,
    "ASCH": 1,
    "LSIL": 1,
    "HSIL": 1,
    "CA": 1,
}

riva_learning_df["binary_label"] = (
    riva_learning_df["class_bethesda"]
    .map(RIVA_BINARY_MAP_PHASE4)
)

# Add a stable dataset identifier
riva_learning_df.insert(
    0,
    "dataset",
    "RIVA"
)

# Rename coordinates to explicitly indicate normalized 0–100 coordinates
riva_learning_df = riva_learning_df.rename(
    columns={
        "nucleus_x": "x_norm",
        "nucleus_y": "y_norm"
    }
)

print("Shape:", riva_learning_df.shape)

print("\nColumns:")
print(riva_learning_df.columns.tolist())

print("\nBinary label counts:")
print(
    riva_learning_df["binary_label"]
    .value_counts()
    .sort_index()
)

print("\nUnmapped labels:")
print(
    riva_learning_df.loc[
        riva_learning_df["binary_label"].isna(),
        "class_bethesda"
    ].unique()
)

display(riva_learning_df.head())

Shape: (15949, 8)

Columns:
['dataset', 'image_filename', 'x_norm', 'y_norm', 'class_bethesda', 'class_annotated', 'cluster_idx', 'binary_label']

Binary label counts:
binary_label
0    10517
1     5432
Name: count, dtype: int64

Unmapped labels:
[]


,dataset,image_filename,x_norm,y_norm,class_bethesda,class_annotated,cluster_idx,binary_label
0,RIVA,LSIL_45_3.png,1000.549618,526.330789,ASCUS,ASCUS,40000.0,1
1,RIVA,LSIL_45_3.png,974.493639,534.147583,ASCUS,ASCUS,40001.0,1
2,RIVA,LSIL_45_3.png,849.424936,138.096692,ASCUS,ASCUS,40002.0,1
3,RIVA,LSIL_45_3.png,839.002545,166.758270,ASCUS,ASCUS,40003.0,1
4,RIVA,LSIL_45_3.png,599.287532,380.417303,INFL,INFL,40004.0,0


In [ ]:
# # ============================================================
# # PHASE 6 — RIVA PREPROCESSING
# # Correct pixel coordinates + cell-centered crops
# # ============================================================

# from pathlib import Path
# from PIL import Image
# import pandas as pd
# import numpy as np

# # ------------------------------------------------------------
# # Paths
# # ------------------------------------------------------------
# RIVA_CROP_ROOT = PROJECT_ROOT / "data" / "processed" / "riva_crops"
# RIVA_CROP_ROOT.mkdir(parents=True, exist_ok=True)

# RIVA_MANIFEST = (
#     PROJECT_ROOT
#     / "data"
#     / "processed"
#     / "riva_learning_units.csv"
# )

# CROP_SIZE = 128
# HALF_CROP = CROP_SIZE // 2

# # ------------------------------------------------------------
# # Start from Phase 5 standardized learning units
# # ------------------------------------------------------------
# riva_samples_df = riva_learning_df.copy()

# # IMPORTANT:
# # Official processed RIVA coordinates are already PIXEL coordinates.
# # Do NOT divide/multiply by 100.
# riva_samples_df["x_pixel"] = (
#     riva_samples_df["x_norm"].round().astype(int)
# )

# riva_samples_df["y_pixel"] = (
#     riva_samples_df["y_norm"].round().astype(int)
# )

# # ------------------------------------------------------------
# # Generate crops image-by-image
# # ------------------------------------------------------------
# crop_paths = {}
# padded_count = 0

# for image_filename, group in riva_samples_df.groupby("image_filename"):

#     image_path = RIVA_IMAGES / image_filename

#     with Image.open(image_path) as img:
#         img = img.convert("RGB")
#         width, height = img.size

#         for idx, row in group.iterrows():

#             x = int(row["x_pixel"])
#             y = int(row["y_pixel"])

#             # Desired crop centered on the cell
#             left = x - HALF_CROP
#             top = y - HALF_CROP
#             right = left + CROP_SIZE
#             bottom = top + CROP_SIZE

#             # Check whether padding is required
#             needs_padding = (
#                 left < 0 or
#                 top < 0 or
#                 right > width or
#                 bottom > height
#             )

#             # Valid source region inside image
#             src_left = max(0, left)
#             src_top = max(0, top)
#             src_right = min(width, right)
#             src_bottom = min(height, bottom)

#             # Fixed-size canvas
#             crop = Image.new(
#                 "RGB",
#                 (CROP_SIZE, CROP_SIZE)
#             )

#             # Position inside canvas
#             dst_left = src_left - left
#             dst_top = src_top - top

#             source_crop = img.crop(
#                 (
#                     src_left,
#                     src_top,
#                     src_right,
#                     src_bottom
#                 )
#             )

#             crop.paste(
#                 source_crop,
#                 (dst_left, dst_top)
#             )

#             # Save
#             stem = Path(image_filename).stem

#             crop_filename = (
#                 f"{stem}_cluster_{int(row['cluster_idx'])}.png"
#             )

#             crop_path = RIVA_CROP_ROOT / crop_filename
#             crop.save(crop_path)

#             crop_paths[idx] = (
#                 str(crop_path.relative_to(PROJECT_ROOT))
#             )

#             if needs_padding:
#                 padded_count += 1

# # ------------------------------------------------------------
# # Add preprocessing information
# # ------------------------------------------------------------
# riva_samples_df["crop_path"] = (
#     riva_samples_df.index.map(crop_paths)
# )

# riva_samples_df["crop_valid"] = (
#     riva_samples_df["crop_path"].notna()
# )

# # ------------------------------------------------------------
# # Save corrected manifest
# # ------------------------------------------------------------
# riva_samples_df.to_csv(
#     RIVA_MANIFEST,
#     index=False
# )

# # ------------------------------------------------------------
# # Essential validation
# # ------------------------------------------------------------
# print("PHASE 6 — RIVA PREPROCESSING")
# print("=" * 55)

# print(f"Learning units:          {len(riva_samples_df):,}")
# print(
#     f"Unique parent images:    "
#     f"{riva_samples_df['image_filename'].nunique():,}"
# )

# print(f"Crop size:               {CROP_SIZE} x {CROP_SIZE}")

# print(
#     f"Valid crops:             "
#     f"{riva_samples_df['crop_valid'].sum():,}"
# )

# print(
#     f"Invalid crops:           "
#     f"{(~riva_samples_df['crop_valid']).sum():,}"
# )

# print(f"Crops requiring padding: {padded_count:,}")

# print("\nPixel coordinate range:")
# print(
#     f"x_pixel: "
#     f"{riva_samples_df['x_pixel'].min()} "
#     f"to {riva_samples_df['x_pixel'].max()}"
# )
# print(
#     f"y_pixel: "
#     f"{riva_samples_df['y_pixel'].min()} "
#     f"to {riva_samples_df['y_pixel'].max()}"
# )

# print("\nBinary labels:")
# print(
#     riva_samples_df["binary_label"]
#     .value_counts()
#     .sort_index()
# )

# print("\nManifest:")
# print(RIVA_MANIFEST)

# print("\nCrops:")
# print(RIVA_CROP_ROOT)

PHASE 6 — RIVA PREPROCESSING
Learning units:          15,949
Unique parent images:    959
Crop size:               128 x 128
Valid crops:             15,949
Invalid crops:           0
Crops requiring padding: 2,650

Pixel coordinate range:
x_pixel: 2 to 1020
y_pixel: 0 to 1016

Binary labels:
binary_label
0    10517
1     5432
Name: count, dtype: int64

Manifest:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\riva_learning_units.csv

Crops:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\riva_crops


In [45]:
# ============================================================
# PHASE 6 — COORDINATE VALIDATION
# Check the actual coordinate format before generating crops
# ============================================================

# Inspect standardized Phase 5 coordinates
print("Phase 5 coordinate ranges")
print("=" * 50)

print("riva_learning_df:")
print(f"x_norm min/max: {riva_learning_df['x_norm'].min()} / "
      f"{riva_learning_df['x_norm'].max()}")
print(f"y_norm min/max: {riva_learning_df['y_norm'].min()} / "
      f"{riva_learning_df['y_norm'].max()}")

print("\nOfficial processed coordinates:")
print(f"nucleus_x min/max: {riva_processed_df['nucleus_x'].min()} / "
      f"{riva_processed_df['nucleus_x'].max()}")
print(f"nucleus_y min/max: {riva_processed_df['nucleus_y'].min()} / "
      f"{riva_processed_df['nucleus_y'].max()}")

# Check actual image dimensions
sample_images = riva_learning_df["image_filename"].drop_duplicates().head(10)

dimensions = []

for filename in sample_images:
    with Image.open(RIVA_IMAGES / filename) as img:
        dimensions.append((filename, img.size))

print("\nSample RIVA image dimensions:")
for filename, size in dimensions:
    print(f"{filename}: {size}")

# Check whether Phase 5 coordinates are actually within 0-100
x_outside = (
    (riva_learning_df["x_norm"] < 0) |
    (riva_learning_df["x_norm"] > 100)
).sum()

y_outside = (
    (riva_learning_df["y_norm"] < 0) |
    (riva_learning_df["y_norm"] > 100)
).sum()

print("\nCoordinate validity if interpreted as 0–100 normalized values:")
print(f"x outside [0,100]: {x_outside:,}")
print(f"y outside [0,100]: {y_outside:,}")

Phase 5 coordinate ranges
riva_learning_df:
x_norm min/max: 2.290827740492169 / 1020.4090620974836
y_norm min/max: 0.0 / 1016.1832061068702

Official processed coordinates:
nucleus_x min/max: 2.290827740492169 / 1020.4090620974836
nucleus_y min/max: 0.0 / 1016.1832061068702

Sample RIVA image dimensions:
LSIL_45_3.png: (1024, 1024)
HSIL_11_5.png: (1024, 1024)
HSIL_22_6.png: (1024, 1024)
LSIL_35_1.png: (1024, 1024)
HSIL_22_10.png: (1024, 1024)
LSIL_29_1.png: (1024, 1024)
HSIL_1_18.png: (1024, 1024)
HSIL_LSIL_2_6.png: (1024, 1024)
LSIL_52_5.png: (1024, 1024)
HSIL_18_13.png: (1024, 1024)

Coordinate validity if interpreted as 0–100 normalized values:
x outside [0,100]: 14,610
y outside [0,100]: 14,484


In [ ]:
# # ============================================================
# # PHASE 6 — CRIC PREPROCESSING
# # Cell-centered crop generation
# # ============================================================

# from pathlib import Path
# from PIL import Image
# import pandas as pd
# import numpy as np

# # ------------------------------------------------------------
# # Paths
# # ------------------------------------------------------------
# CRIC_CROP_ROOT = (
#     PROJECT_ROOT
#     / "data"
#     / "processed"
#     / "cric_crops"
# )

# CRIC_CROP_ROOT.mkdir(parents=True, exist_ok=True)

# CRIC_MANIFEST = (
#     PROJECT_ROOT
#     / "data"
#     / "processed"
#     / "cric_learning_units.csv"
# )

# CROP_SIZE = 128
# HALF_CROP = CROP_SIZE // 2

# # ------------------------------------------------------------
# # Load CRIC cell annotations
# # ------------------------------------------------------------
# cric_df = pd.read_csv(
#     CRIC_METADATA / "classifications.csv"
# )

# # ------------------------------------------------------------
# # Binary mapping from Phase 4
# # ------------------------------------------------------------
# CRIC_BINARY_MAP = {
#     "Negative for intraepithelial lesion": 0,
#     "ASC-US": 1,
#     "ASC-H": 1,
#     "LSIL": 1,
#     "HSIL": 1,
#     "SCC": 1,
# }

# # ------------------------------------------------------------
# # Create standardized CRIC learning-unit table
# # ------------------------------------------------------------
# cric_samples_df = pd.DataFrame({
#     "dataset": "CRIC",
#     "image_filename": cric_df["image_filename"],
#     "cell_id": cric_df["cell_id"],
#     "x_pixel": cric_df["nucleus_x"].round().astype(int),
#     "y_pixel": cric_df["nucleus_y"].round().astype(int),
#     "class_bethesda": cric_df["bethesda_system"],
# })

# cric_samples_df["binary_label"] = (
#     cric_samples_df["class_bethesda"]
#     .map(CRIC_BINARY_MAP)
# )

# # Verify that every label mapped
# if cric_samples_df["binary_label"].isna().any():
#     unmapped = (
#         cric_samples_df.loc[
#             cric_samples_df["binary_label"].isna(),
#             "class_bethesda"
#         ]
#         .unique()
#         .tolist()
#     )
#     raise ValueError(f"Unmapped CRIC labels: {unmapped}")

# # ------------------------------------------------------------
# # Generate crops image-by-image
# # ------------------------------------------------------------
# crop_paths = {}
# padded_count = 0

# for image_filename, group in cric_samples_df.groupby(
#     "image_filename"
# ):

#     image_path = CRIC_IMAGES / image_filename

#     with Image.open(image_path) as img:
#         img = img.convert("RGB")
#         width, height = img.size

#         for idx, row in group.iterrows():

#             x = int(row["x_pixel"])
#             y = int(row["y_pixel"])

#             # Desired 128x128 crop centered on nucleus
#             left = x - HALF_CROP
#             top = y - HALF_CROP
#             right = left + CROP_SIZE
#             bottom = top + CROP_SIZE

#             needs_padding = (
#                 left < 0
#                 or top < 0
#                 or right > width
#                 or bottom > height
#             )

#             # Clip source region to image boundaries
#             src_left = max(0, left)
#             src_top = max(0, top)
#             src_right = min(width, right)
#             src_bottom = min(height, bottom)

#             # Fixed-size output canvas
#             crop = Image.new(
#                 "RGB",
#                 (CROP_SIZE, CROP_SIZE)
#             )

#             # Destination position on canvas
#             dst_left = src_left - left
#             dst_top = src_top - top

#             source_crop = img.crop(
#                 (
#                     src_left,
#                     src_top,
#                     src_right,
#                     src_bottom
#                 )
#             )

#             crop.paste(
#                 source_crop,
#                 (dst_left, dst_top)
#             )

#             # Unique crop filename
#             stem = Path(image_filename).stem

#             crop_filename = (
#                 f"{stem}_cell_{int(row['cell_id'])}.png"
#             )

#             crop_path = CRIC_CROP_ROOT / crop_filename
#             crop.save(crop_path)

#             crop_paths[idx] = (
#                 str(crop_path.relative_to(PROJECT_ROOT))
#             )

#             if needs_padding:
#                 padded_count += 1

# # ------------------------------------------------------------
# # Add crop information
# # ------------------------------------------------------------
# cric_samples_df["crop_path"] = (
#     cric_samples_df.index.map(crop_paths)
# )

# cric_samples_df["crop_valid"] = (
#     cric_samples_df["crop_path"].notna()
# )

# # ------------------------------------------------------------
# # Save manifest
# # ------------------------------------------------------------
# cric_samples_df.to_csv(
#     CRIC_MANIFEST,
#     index=False
# )

# # ------------------------------------------------------------
# # Essential validation
# # ------------------------------------------------------------
# print("PHASE 6 — CRIC PREPROCESSING")
# print("=" * 55)

# print(
#     f"Learning units:          "
#     f"{len(cric_samples_df):,}"
# )

# print(
#     f"Unique parent images:    "
#     f"{cric_samples_df['image_filename'].nunique():,}"
# )

# print(f"Crop size:               {CROP_SIZE} x {CROP_SIZE}")

# print(
#     f"Valid crops:             "
#     f"{cric_samples_df['crop_valid'].sum():,}"
# )

# print(
#     f"Invalid crops:           "
#     f"{(~cric_samples_df['crop_valid']).sum():,}"
# )

# print(
#     f"Crops requiring padding: "
#     f"{padded_count:,}"
# )

# print("\nPixel coordinate range:")
# print(
#     f"x_pixel: "
#     f"{cric_samples_df['x_pixel'].min()} "
#     f"to {cric_samples_df['x_pixel'].max()}"
# )

# print(
#     f"y_pixel: "
#     f"{cric_samples_df['y_pixel'].min()} "
#     f"to {cric_samples_df['y_pixel'].max()}"
# )

# print("\nBinary labels:")
# print(
#     cric_samples_df["binary_label"]
#     .value_counts()
#     .sort_index()
# )

# print("\nManifest:")
# print(CRIC_MANIFEST)

# print("\nCrops:")
# print(CRIC_CROP_ROOT)

PHASE 6 — CRIC PREPROCESSING
Learning units:          11,534
Unique parent images:    400
Crop size:               128 x 128
Valid crops:             11,534
Invalid crops:           0
Crops requiring padding: 885

Pixel coordinate range:
x_pixel: 3 to 1278
y_pixel: 1 to 1019

Binary labels:
binary_label
0    6779
1    4755
Name: count, dtype: int64

Manifest:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_learning_units.csv

Crops:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_crops
